Propósito: Filtra las empresas del SPY500 para traer los papeles en sobreventa. Usa parametros de los indicadores RSI, MACD y Koncorde en la temporalidad de 60min y 7días.

Este filtro busca capturar movimientos que duran entre 2 a 5 días.

In [1]:
%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [3]:
#1. Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = df_spy['Symbol'].tolist()

print(f"Se han cargado {len(tickers_a_analizar)} activos del SPY.")
print("tickers_a_analizar:", tickers_a_analizar[:600])

Se han cargado 503 activos del SPY.
tickers_a_analizar: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP', 'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK.B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BE', 'BNY', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF.B', 'BG', 'BXP', 'CHRW', 'CDNS', 'CPT', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CASY', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CIEN', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COHR', 'COIN', 'CL', 'CMCSA', 'FIX', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CSGP', 'COST', '

In [4]:
# def calcular_koncorde(df):

    #rango_min = df['Low'].rolling(window=14).min()
    #rango_max = df['High'].rolling(window=14).max()

    #mano_debil = ((df['Close'] - rango_min) / (rango_max - rango_min) * 100) - 50
    #return mano_debil.iloc[-1], mano_debil.iloc[-2]

# Limpiar tickets (Si no existen, usamos una lista pequeña de prueba)
try:
    tickers_limpios = [t.replace('.', '-') for t in tickers_a_analizar]
except NameError:
    print("La lista 'tickers_a_analizar' no existía. Usando lista de prueba...")
    tickers_limpios = ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'TSLA', 'META', 'NVDA']

# Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_limpios.extend(mis_tickers)

# DESCARGA Y ANÁLISIS.
print(f"Descargando datos para {len(tickers_limpios)} activos...")
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        # kon_act, kon_ant = calcular_koncorde(df)

        precio_act = df['Close'].iloc[-1]
        precio_ant = df['Close'].iloc[-2]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and macd_act <= sig_act 
            #and kon_act < 0
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'Prec_Act': round(precio_act, 2),
                'Prec_Ant': round(precio_ant, 2),
                'RSI_Act': round(rsi_act, 1),
                'RSI_Ant': round(rsi_ant, 1),
                'RSI_MA': round(rsi_ma, 1),
                
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
                
                #'Konk_Act': round(kon_act, 1),
                #'Konk_Ant': round(kon_ant, 1)
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI_Act').to_string(index=False))
else:
    print("Sin señales detectadas.")

Descargando datos para 520 activos...

 Ticker  Prec_Act  Prec_Ant  RSI_Act  RSI_Ant  RSI_MA     MACD
     MS    180.00    181.22      9.5     10.9    39.5   -1.483
    BNY    136.31    136.35     10.9     10.9    33.6   -1.287
     BX    106.11    107.47     11.9     14.6    36.8   -1.310
    URA     35.83     36.06     14.0     15.4    25.1   -0.685
    BAC     50.10     49.88     15.4     11.4    50.3   -0.526
    KKR     84.57     85.59     15.9     18.9    37.2   -1.162
    UAL     99.22    100.10     16.4     18.7    28.5   -1.824
    CRH     78.94     79.51     16.7     21.0    33.0   -0.557
   CDNS    331.00    332.47     16.8     18.3    27.8   -5.209
   PANW    379.46    381.96     17.2     18.6    36.3   -4.918
    IVZ     28.01     28.30     17.2     20.7    28.1   -0.391
      C    123.07    123.02     17.5     16.9    47.4   -1.163
   DELL    521.90    526.35     18.2     19.4    42.3   -7.540
    STT    160.95    159.52     18.3     11.5    40.4   -2.209
    NEM    111.5

In [5]:
print(len(tickers_limpios))
print(tickers_limpios)
print(len(tickers_a_analizar))
print(len(empresas_60min))

520
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP', 'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BE', 'BNY', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BG', 'BXP', 'CHRW', 'CDNS', 'CPT', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CASY', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CIEN', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COHR', 'COIN', 'CL', 'CMCSA', 'FIX', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CSGP', 'COST', 'CRH', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DR

In [6]:
# DESCARGA Y ANÁLISIS SEMANAL
print(f"Descargando datos SEMANALES para {len(tickers_limpios)} activos...")
data_7d = yf.download(tickers_limpios, period="2y", interval="1wk", group_by='ticker', progress=False)

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()

        if len(df) < 50: continue

        # Cálculos Técnicos (La misma lógica, distinta escala temporal)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        
        #kon_act, kon_ant = calcular_koncorde(df)

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO MACRO
        if rsi_act < 40 and macd_act < sig_act and kon_act < 0:
            empresas_semanal.append({
                'Ticker': ticker,
                'Prec_Seman': round(precio_act, 2),
                'RSI_Sem': round(rsi_act, 1),
                'RSI_MA_S': round(df['RSI_MA'].iloc[-1], 1),
                'Konk_S_Act': round(kon_act, 1),
                'Konk_S_Ant': round(kon_ant, 1)
            })
    except:
        continue

print("\n" + "="*95)


print("GANGAS DETECTADAS EN SEMANAL (VISIÓN MACRO):")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    # Ordenamos por Koncorde para ver dónde el pánico semanal es más extremo
    print(df_final.sort_values(by='RSI_Sem').to_string(index=False))
else:
    print("Sin señales semanales.")

Descargando datos SEMANALES para 520 activos...

GANGAS DETECTADAS EN SEMANAL (VISIÓN MACRO):
Sin señales semanales.


In [7]:
# FRACTALES ALINEADOS, CRUCE DE 60 MINUTOS AND SEMANAL.

# Convertimos las listas de diccionarios en DataFrames (si no lo eran ya)
df_60 = pd.DataFrame(empresas_60min)
df_sem = pd.DataFrame(empresas_semanal)
print("Alineando las temporalidades.")
if not df_60.empty and not df_sem.empty:
    # Solo quedarán los activos que estén presentes en AMBOS DataFrames
    gangas_alineadas = pd.merge(df_60, df_sem, on='Ticker')


    if not gangas_alineadas.empty:
        print("\n" + "!"*30)
        print("ALINEACIÓN DE TEMPORALIDAD DETECTADA")
        print("Activos en zona de pánico tanto en 60min como en 7d:")
        # Seleccionamos las columnas más relevantes para la comparativa
        columnas_finales = [
            'Ticker', 'Prec_Act', 'RSI_Act', 'Konk_Act',
            'RSI_Sem', 'Konk_S_Act'
        ]
        print(gangas_alineadas[columnas_finales].to_string(index=False))

        print("\nInterpretación: El activo está en suelo macro y dando señal de giro micro.")
    else:
        print("\nNo hay coincidencias 'AND'.")
        print("Existen gangas individuales, pero no están alineadas en ambos fractales hoy.")
else:
    if df_60.empty:
        print("- El fractal de 60 min no detectó señales.")
    if df_sem.empty:
        print("- El fractal Semanal no detectó señales.")
    print("\nNo se puede realizar la operación AND sin datos en ambos marcos.")

Alineando las temporalidades.
- El fractal Semanal no detectó señales.

No se puede realizar la operación AND sin datos en ambos marcos.


In [8]:
# SEMÁFORO Y REVERSIÓN DE CALIDAD

UMBRAL_PANICO = 35       # Para activos "estirados" al extremo (Cuchillos)
UMBRAL_CALIDAD = 48      # Para captar reversiones inminentes en líderes
EMPRESAS_FILTRADAS = []

print(f"Iniciando escáner sobre {len(tickers_limpios)} activos...")

for ticker in tickers_limpios:
    try:
        # Reutilizamos los datos de la celda de 60 min (Micro Fractal)
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos (Referenciando el fractal de 1h)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)

        # Variables de control
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        kon_act, kon_ant = calcular_koncorde(df)

        # FILTRO ESTIRADO (Filtro de Entrada)
        # Captura el pánico real O sobreventa relativa en empresas fuertes
        if rsi_act < UMBRAL_CALIDAD and kon_act < 0:

            # LÓGICA DEL SEMÁFORO (El Gatillo)
            # Caso Verde: Cruce de Media al alza (Ayer < MA y Hoy > MA)
            if rsi_act > rsi_ma and rsi_ant <= rsi_ma:
                estado = "🟢 GATILLO (Cruce confirmado)"
            # Caso Amarillo: Frenado de caída (Ayer < Hoy pero aún debajo de MA)
            elif rsi_act > rsi_ant:
                estado = "🟡 AGOTAMIENTO (En observación)"
            # Caso Rojo: Inercia bajista (Sigue cayendo)
            else:
                estado = "🔴 CUCHILLO (Sin señal de giro)"

            # Buscamos si el ticker también dio señal en la lista semanal previa
            es_ganga_macro = any(e['Ticker'] == ticker for e in empresas_semanal)
            etiqueta_macro = "🔥 ALINEADO (Macro+Micro)" if es_ganga_macro else "Solo Micro"

            EMPRESAS_FILTRADAS.append({
                'Ticker': ticker,
                'Estado': estado,
                'RSI': round(rsi_act, 1),
                'RSI_MA': round(rsi_ma, 1),
                'Tipo': "Extremo" if rsi_act < UMBRAL_PANICO else "Calidad",
                'Fractal': etiqueta_macro
            })

    except Exception:
        continue

# PRESENTACIÓN DE RESULTADOS
print("\n" + "="*100)
print(f"REPORTE DEL S&P500 - {len(EMPRESAS_FILTRADAS)} ACTIVOS EN RADAR")
print("="*100)

if EMPRESAS_FILTRADAS:
    df_final = pd.DataFrame(EMPRESAS_FILTRADAS)

    # Ordenamos por prioridad: Gatillos primero, luego Agotamiento, luego Cuchillos
    orden_estado = {"🟢 GATILLO (Cruce confirmado)": 0, "🟡 AGOTAMIENTO (En observación)": 1, "🔴 CUCHILLO CAYENDO (Sin señal de giro)": 2}
    df_final['Sort'] = df_final['Estado'].map(orden_estado)
    df_final = df_final.sort_values(by=['Sort', 'RSI']).drop(columns='Sort')

    print(df_final.to_string(index=False))
else:
    print("El mercado no presenta activos por debajo de RSI 48 con pánico de mano débil.")

Iniciando escáner sobre 520 activos...

REPORTE DEL S&P500 - 0 ACTIVOS EN RADAR
El mercado no presenta activos por debajo de RSI 48 con pánico de mano débil.
